In [ ]:
import os
import random
import torch

from datasets import Dataset, load_from_disk
from sentence_transformers import InputExample, SentenceTransformer, SentenceTransformerTrainer, SentenceTransformerTrainingArguments, losses
from sklearn.model_selection import train_test_split

In [2]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [4]:
wiki_corpus = load_from_disk("wiki_corpus_data")['passages']
wiki_test_set = load_from_disk("wiki_test_set_data")['test']

bio_corpus = load_from_disk("bio_corpus_data")['passages']
bio_test_set = load_from_disk("bio_test_set_data")['test']

In [5]:
wiki_id_to_passage = {entry["id"]: entry["passage"] for entry in wiki_corpus}

wiki_passages = [c['passage'] for c in wiki_corpus]

wiki_samples = []

for example in wiki_test_set:
    query = example["question"]

    positive_passage = wiki_id_to_passage[int(example['id'])]

    if not positive_passage:
        continue  # Skip if no relevant passages found
    
    # Choose a random negative passage that is *not* in the relevant set
    negative_passage = random.choice(wiki_passages)
    while negative_passage in positive_passage:
        negative_passage = random.choice(wiki_passages)
        
    wiki_samples.append(InputExample(texts=[query, positive_passage], label=1.0))
    wiki_samples.append(InputExample(texts=[query, negative_passage], label=0.0))

print(f"Generated {len(wiki_samples)} wiki samples")

Generated 1836 wiki samples


In [6]:
bio_id_to_passage = {entry["id"]: entry["passage"] for entry in bio_corpus}

bio_passages = [c['passage'] for c in bio_corpus]

bio_samples = []

for example in bio_test_set:
    query = example["question"]

    relevant_passages = []
    for passage_id in example['relevant_passage_ids'][1:-1].split(','):
        relevant_passages.append(bio_id_to_passage[int(passage_id)])

    if not relevant_passages:
        continue  # Skip if no relevant passages found

    positive_passage = random.choice(relevant_passages)
    
    # Choose a random negative passage that is *not* in the relevant set
    negative_passage = random.choice(bio_passages)
    while negative_passage in positive_passage:
        negative_passage = random.choice(bio_passages)
        
    bio_samples.append(InputExample(texts=[query, positive_passage], label=1.0))
    bio_samples.append(InputExample(texts=[query, negative_passage], label=0.0))

print(f"Generated {len(bio_samples)} bio samples")

Generated 9438 bio samples


In [7]:
combined_samples = wiki_samples + bio_samples
print(f"Generated {len(combined_samples)} combined samples")

Generated 11274 combined samples


In [8]:
# Function to split a dataset into training and evaluation sets
def split_dataset(sample_data, test_size=0.2, random_state=42):
    train_data, eval_data = train_test_split(sample_data, test_size=test_size, random_state=random_state)
    train_dataset = Dataset.from_dict({
        "query": [ex.texts[0] for ex in train_data],
        "passage": [ex.texts[1] for ex in train_data],
        "label": [ex.label for ex in train_data]
    })
    eval_dataset = Dataset.from_dict({
        "query": [ex.texts[0] for ex in eval_data],
        "passage": [ex.texts[1] for ex in eval_data],
        "label": [ex.label for ex in eval_data]
    })
    return train_dataset, eval_dataset

In [9]:
# Split each dataset into training and evaluation sets
wiki_train_dataset, wiki_eval_dataset = split_dataset(wiki_samples)
bio_train_dataset, bio_eval_dataset = split_dataset(bio_samples)
combined_train_dataset, combined_eval_dataset = split_dataset(combined_samples)

# Print dataset sizes to verify the splits
print(f"Wiki dataset: {len(wiki_train_dataset)} train, {len(wiki_eval_dataset)} eval")
print(f"Bio dataset: {len(bio_train_dataset)} train, {len(bio_eval_dataset)} eval")
print(f"Combined dataset: {len(combined_train_dataset)} train, {len(combined_eval_dataset)} eval")

Wiki dataset: 1468 train, 368 eval
Bio dataset: 7550 train, 1888 eval
Combined dataset: 9019 train, 2255 eval


In [10]:
# Define the datasets to use
datasets = ['wiki', 'bio', 'combined']

# Define the hyperparameter combinations to try
batch_sizes = [8, 16, 32]
learning_rates = [1e-5, 2e-5, 5e-5]
num_epochs = [10, 25]
warmup_ratios = [0.05, 0.1, 0.2]

# Loop through all combinations of hyperparameters
for dataset_name in datasets:
    print(f'======= Training on {dataset_name} dataset =======')
    
    # Select the appropriate datasets
    if dataset_name == 'wiki':
        train_dataset = wiki_train_dataset
        eval_dataset = wiki_eval_dataset
    elif dataset_name == 'bio':
        train_dataset = bio_train_dataset
        eval_dataset = bio_eval_dataset
    elif dataset_name == 'combined':
        train_dataset = combined_train_dataset
        eval_dataset = combined_eval_dataset
    
    # Initialize variables to track the best configuration
    best_val_loss = float('inf')
    best_params = {}
    
    for batch_size, learning_rate, num_train_epochs, warmup_ratio in zip(batch_sizes, learning_rates, num_epochs, warmup_ratios):
        print(f"Training with batch_size={batch_size}, learning_rate={learning_rate}, num_train_epochs={num_train_epochs}, warmup_ratio={warmup_ratio}")
        
        # Load model
        model = SentenceTransformer("thenlper/gte-small", device=device)
        
        # Define loss function
        train_loss = losses.CosineSimilarityLoss(model)
        
        # Define training arguments with evaluation strategy
        args = SentenceTransformerTrainingArguments(
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            num_train_epochs=num_train_epochs,
            learning_rate=learning_rate,
            warmup_ratio=warmup_ratio,
            eval_strategy="epoch",
            save_strategy="no",
            load_best_model_at_end=False,
            save_total_limit=0,
            metric_for_best_model=None
        )
        
        # Initialize trainer with both training and validation datasets
        trainer = SentenceTransformerTrainer(
            model=model,
            args=args,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            loss=train_loss
        )
        
        # Train the model
        trainer.train()
        
        # Get the final validation loss
        val_loss = None
        for log_entry in trainer.state.log_history:
            if 'eval_loss' in log_entry:
                val_loss = log_entry['eval_loss']
        print(f"Final validation loss: {val_loss}")
        
        # Check if this configuration is the best so far
        if val_loss < best_val_loss or best_val_loss == float('inf'):
            if val_loss < best_val_loss:
                best_val_loss = val_loss
            best_params = {
                "batch_size": batch_size,
                "learning_rate": learning_rate,
                "num_train_epochs": num_train_epochs,
                "warmup_ratio": warmup_ratio
            }
    
    # Output the best configuration
    print(f"Best hyperparameters for {dataset_name} dataset: {best_params} with validation loss: {best_val_loss}")
    
    # Train final model with the best hyperparameters
    final_model = SentenceTransformer("thenlper/gte-small", device=device)
    
    # Define final loss function
    final_loss = losses.CosineSimilarityLoss(final_model)
    
    # Define final training arguments
    final_args = SentenceTransformerTrainingArguments(
        per_device_train_batch_size=best_params["batch_size"],
        per_device_eval_batch_size=best_params["batch_size"],
        num_train_epochs=best_params["num_train_epochs"],
        learning_rate=best_params["learning_rate"],
        warmup_ratio=best_params["warmup_ratio"],
        eval_strategy="epoch",
        save_strategy="no",
        save_total_limit=0
    )
    
    print(f'Retraining on {dataset_name} dataset with best hyperparameters')
    
    # Final training
    final_trainer = SentenceTransformerTrainer(
        model=final_model,
        args=final_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        loss=final_loss
    )
    
    final_trainer.train()
    
    # Save the fine-tuned model
    final_model.save(f"retrieval_finetuned_gte_small_best_{dataset_name}")

======= Training on wiki dataset =======
Training with batch_size=8, learning_rate=1e-05, num_train_epochs=10, warmup_ratio=0.05


Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Epoch,Training Loss,Validation Loss
1,No log,0.246960
2,No log,0.237534
3,0.246100,0.227548
4,0.246100,0.214454
5,0.246100,0.204618
6,0.174500,0.197790
7,0.174500,0.194417
8,0.174500,0.193178
9,0.132500,0.193233
10,0.132500,0.192857


Final validation loss: 0.1928572803735733
Training with batch_size=16, learning_rate=2e-05, num_train_epochs=25, warmup_ratio=0.1


...

Final validation loss: 0.20097753405570984
Best hyperparameters for combined dataset: {'batch_size': 8, 'learning_rate': 1e-05, 'num_train_epochs': 10, 'warmup_ratio': 0.05} with validation loss: 0.15574616193771362


Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Retraining on combined dataset with best hyperparameters


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Epoch,Training Loss,Validation Loss
1,0.156800,0.143679
2,0.138700,0.139353
3,0.122500,0.138870
4,0.109800,0.136599
5,0.102400,0.138789
6,0.091500,0.142170
7,0.084200,0.149577
8,0.081100,0.154101
9,0.075400,0.152234
10,0.071500,0.155745
